

**Nombre y No control:**

* Brayan Gres Juarez
* 23111409


In [1]:
#--------------------------------------------------------------------
# Incluye aquí todas las librerías y paquetes que requieras.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#-------------------------------------------------------------------

#Ejercicio
from sklearn.decomposition import TruncatedSVD

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **Liga de Datos de la UCI "Restaurant & consumer data".**
### **Descarga el archivo "restaurant+consumer+data.zip" y de ahí necesitaremos los archivos "rating_final.csv" y "geoplaces2.csv".**

https://archive.ics.uci.edu/dataset/232/restaurant+consumer+data



In [3]:
# Descarga los archivos de la página de la UCI a partir de los cuales generaremos
# nuestras matrices de utilidad.

data1 = pd.read_csv("/content/drive/MyDrive/ArchivosAnalitica/SistemaDeRecomendacion/rating_final.csv", header='infer', sep=",")
data2 = pd.read_csv("/content/drive/MyDrive/ArchivosAnalitica/SistemaDeRecomendacion/geoplaces2.csv", header='infer',  encoding='latin-1')

print(data1.shape, data2.shape)


(1161, 5) (130, 21)


**NOTA: Recuerda que cada vez que se te pida incluir líneas de código, podrás incluir 1 o más líneas, pero siempre en la sección donde se te indica hacerlo.**

## **Ejercicio - 1**

### **Explica cuál es el propósito de usar el argumento "latin-1" al cargar el segundo archivo.**

++++++++ Inicia la sección de agregar texto: +++++++++++

encoding='latin-1'le indica a pandas con que codificacion de caracteres esta guardado el archivo geoplaces2.csv.
Ya que por defecto , pd.read_csv lee lso archivos como UTF-8. Pero el archivo esta guardado como latin-1, y trae nombres de restaurantes, calles y ciudades de México, incluye caracteres especiales del español como acentos.
Con encoding='latin-1' pandas interpreta bien cada byte, el archivo se carga sin errores y el texto se conserva correctamente.

++++++++ Termina la sección de agregar texto. +++++++++++

In [4]:
# Del primer archivo obtenemos una matriz con 3 evaluaciones de los restaurantes:
# general, comida y servicio.
# Las evaluaciones pueden ser 0, 1 o 2, siendo 0 la menor calificación y 2 la
# mayor calificación:

data1.head()

,userID,placeID,rating,food_rating,service_rating
0,U1077,135085,2,2,2
1,U1077,135038,2,2,1
2,U1077,132825,2,2,2
3,U1077,135060,1,2,2
4,U1068,135104,1,1,2


In [5]:
# Del segundo archivo obtenemos información diversa de cada restaurante:

data2.head(2)

,placeID,latitude,longitude,the_geom_meter,name,address,city,state,country,fax,...,alcohol,smoking_area,dress_code,accessibility,price,url,Rambience,franchise,area,other_services
0,134999,18.915421,-99.184871,0101000020957F000088568DE356715AC138C0A525FC46...,Kiku Cuernavaca,Revolucion,Cuernavaca,Morelos,Mexico,?,...,No_Alcohol_Served,none,informal,no_accessibility,medium,kikucuernavaca.com.mx,familiar,f,closed,none
1,132825,22.147392,-100.983092,0101000020957F00001AD016568C4858C1243261274BA5...,puesto de tacos,esquina santos degollado y leon guzman,s.l.p.,s.l.p.,mexico,?,...,No_Alcohol_Served,none,informal,completely,low,?,familiar,f,open,none


In [6]:
# De data1 no requeriremos "rating" y de data2 solo necesitamos "placeID" y "name":
# Definimos la lista y matrices con los factores que necesitamos:
# ************* Inlcuye aquí tu código:*****************************
lista_data1 = ['userID', 'placeID', 'food_rating','service_rating']
lista_data2 = ['placeID','name']
# *********** Aquí termina la sección de agregar código *************
data1a = data1[lista_data1]
data2a = data2[lista_data2]

## **Ejercicio - 2**

In [7]:
# Define el DataFrame que conjunte la información de las dos DataFrame data1a
# y data2a en uno solo, a través de la columna común "placeID" y que
# llamaremos "df_combinado":

# ************* Inlcuye aquí tu código:*****************************
#utilice  merge()
df_combinado = pd.merge(data1a, data2a, on="placeID")

# *********** Aquí termina la sección de agregar código *************

# Despleguemos la dimensión y los primeros renglones de este DataFrame:

print(df_combinado.shape)
df_combinado.head()

(1161, 5)


,userID,placeID,food_rating,service_rating,name
0,U1077,135085,2,2,Tortas Locas Hipocampo
1,U1077,135038,2,1,Restaurant la Chalita
2,U1077,132825,2,2,puesto de tacos
3,U1077,135060,2,2,Restaurante Marisco Sam
4,U1068,135104,1,2,vips


## **Ejercicio - 3**

In [8]:
#    Define la matriz de utilidad cuyos renglones sean los nombres de los
#    restaurantes, las columnas los IDs de los usuarios y las entradas la
#    evaluación de la comida (food_rating). La llamaremos "UtMx_food".


# ************* Inlcuye aquí tu código:*****************************


UtMx_food = df_combinado.pivot_table(values="food_rating", index="name", columns="userID", fill_value=0)


# *********** Aquí termina la sección de agregar código *************


print('Dimensión de la matriz de Utilidad sobre la evaluación de la comida:')
print('(restaurantes, usuarios) =', (UtMx_food.shape))
UtMx_food.head()

Dimensión de la matriz de Utilidad sobre la evaluación de la comida:
(restaurantes, usuarios) = (129, 138)


userID,U1001,U1002,U1003,U1004,U1005,U1006,U1007,U1008,U1009,U1010,...,U1129,U1130,U1131,U1132,U1133,U1134,U1135,U1136,U1137,U1138
name,,,,,,,,,,,,,,,,,,,,,
Abondance Restaurante Bar,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Arrachela Grill,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cabana Huasteca,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
Cafe Chaires,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cafeteria cenidet,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## **Ejercicio - 4**

### **En la Factorización SVD la cantidad de valores singulares será menor o igual al menor valor de los renglones o columnas de la matriz de utilidad UtMx.**

### **La factorización SVD de una matriz $A$ tiene la forma:**

$A_{m\times n} = U_{m\times m}\Sigma_{m\times n}V_{n\times n}^T$

In [9]:
# a. Aplica la transformación SVD para obtener la matriz de las variables
#    latentes de los restaurantes, en relación a la manera en que los usuarios
#    evaluaron la comida. Deberás obtener la factorización SVD con el máximo número
#    de componentes y por lo tanto el máximo de valores singulares posibles de acuerdo
#    a este criterio. Así, de acuerdo al resultado obtenido en el Ejercicio-3, asigna
#    a la variable "nc_food" el número entero adecuado para llegar a obtener el
#    resultado pedido.


# ************* Inlcuye aquí tu código:**********************************************

#Aplica Factorización SVD
U,s, Vt=np.linalg.svd(UtMx_food,full_matrices=False)
nc_food = min(UtMx_food.shape)    # número de componentes

# *********** Aquí termina la sección de agregar código *****************************

print('Total de valores singulares basados en evaluación de la comida:', nc_food)

Total de valores singulares basados en evaluación de la comida: 129


In [10]:
# b. Lleva a cabo la factorización SVD truncada en la matriz de utilidad
#    obtenida previamente y con respecto al número de componentes dada
#    por la variable nc_food.

# ************* Inlcuye aquí tu código:********************************

nc_food= min(UtMx_food.shape) #Preguntar a la maestra
SVD_food = TruncatedSVD(n_components=nc_food, random_state=1)
SVD_food.fit_transform(UtMx_food)  #transforma

# *********** Aquí termina la sección de agregar código ***************

# Con la factorización SVD truncada obtenida, determinemos la cantidad
# de componentes que explican un 90% de la variabilidad acumulada de
# cada componente:

for j in range(nc_food):
  if SVD_food.explained_variance_ratio_[0:j].sum() > 0.90:
    break

# Usaremos esta cantidad de componentes para las recomendaciones basadas
# en la calificación de la comida:
N_food = j-1

print('Valor de truncamiento al 90% de dicha variabilidad:', N_food)

Valor de truncamiento al 90% de dicha variabilidad: 51


In [11]:
# Usando la métrica de correlación de Pearson, obtenemos ahora las 10 mejores
# recomendaciones de este modelo no supervisado con base a la información de
# alguien que desea obtener las similitudes con el "Restaurante Pueblo Bonito"
# y con la cota de truncamiento obtenido en el ejercicio anterior.
# Para ello obtenemos las 10 mejores correlaciones positivas a continuación.

# Factorización SVD:
SVD_food = TruncatedSVD(n_components = N_food)
resultant_matrix_food = SVD_food.fit_transform(UtMx_food)

# Matriz de correlación de Pearson:
corr_mat_food = np.corrcoef(resultant_matrix_food)

# Restaurante de referencia:
restaurante_de_referencia = "Restaurante Pueblo Bonito"
nombres_rest = UtMx_food.T.columns  # nombres de restaurantes
idx_rest = list(nombres_rest).index(restaurante_de_referencia)
corr_rest = corr_mat_food[idx_rest] # Vector de Correlación del RinconHuasteco

# Buscando las correlaciones positivas:
idx = (corr_rest>0)
mejores_sim_food = list()
for i in range(len(nombres_rest[idx])):
  mejores_sim_food.append((corr_rest[idx][i], nombres_rest[idx][i]))

print('Total de similaridades positivas encontradas:', len(mejores_sim_food))

print('Algunos de los resultados encontrados:')
mejores_sim_food[0:5]

Total de similaridades positivas encontradas: 67
Algunos de los resultados encontrados:


[(np.float64(0.048624059647091725), 'Cabana Huasteca'),
 (np.float64(0.019211671297457603), 'Cafe Chaires'),
 (np.float64(0.007174717428744527), 'Cafeteria cenidet'),
 (np.float64(0.23967874141342543), 'Cafeteria y Restaurant El Pacifico'),
 (np.float64(0.028508354814271386), 'Cenaduria El RincÃ³n de Tlaquepaque')]

## **Ejercicio - 5**

In [12]:
# Ordenar la lista de recomendaciones "mejores_sim_food" encontrada en el paso
# anterior de manera descendente. Llamaremos "mejores_sim_food_ordenadas" a
# dicha lista.



mejores_sim_food_ordenadas = sorted(mejores_sim_food, key=lambda simil: simil[0], reverse=True)



# Desplegamos las 10 mejores similitudes encontradas de manera descendente:
print('Similitudes con base a la evaluación de la comida con mayores valores de correlación:')
for k in range(1,11):
  print('%d> %s' % (k, mejores_sim_food_ordenadas[k]))

Similitudes con base a la evaluación de la comida con mayores valores de correlación:
1> (np.float64(0.7941790171304194), 'Restaurante la Estrella de Dima')
2> (np.float64(0.7036209014674555), 'pizza clasica')
3> (np.float64(0.6543010201788902), 'Restaurante Guerra')
4> (np.float64(0.5563449057129521), 'El Club')
5> (np.float64(0.3841307514988613), 'Pizzeria Julios')
6> (np.float64(0.31138724585344174), 'Hamburguesas saul')
7> (np.float64(0.30462357692471764), 'Restaurant Oriental Express')
8> (np.float64(0.29329725659350714), 'Restaurante El Cielo Potosino')
9> (np.float64(0.29295575606033575), 'La Fontana Pizza Restaurante and Cafe')
10> (np.float64(0.29271841663824677), 'la Cochinita Pibil Restaurante Yucateco')


## **Ejercicio - 6**

### **De manera análoga y usando ahora la evaluación de un restaurante con respecto a su servicio (service_rating), encuentra ahora los diez restaurantes con mayor similaridad al mismo restaurante anterior llamado "Restaurante Pueblo Bonito" y con respecto también a la misma métrica de similaridad de Pearson.**

### **Deberás usar la cantidad de componentes necesarias para que la varabilidad de la factorización SVD quede explicada en un 90%.**

### **a.  Para ello define primero la matriz de utilidad, UtMx_service, donde los renglones son los nombres de los restaurantes, las columnas los usuarios y las entradas las evaluaciones con respecto al servicio recibido en el restaurante.**





In [13]:
# a. Define la matriz de utilidad cuyos renglones sean los nombres de los
#    restaurantes, las columnas los IDs de los usuarios y las entradas la
#    evaluación del servicio (service_rating). La llamaremos "UtMx_service".

# ************* Inlcuye aquí tu código:*****************************

UtMx_service = df_combinado.pivot_table(values="service_rating", index="name", columns="userID", fill_value=0)

# *********** Aquí termina la sección de agregar código *************

print('Dimensión de la matriz de Utilidad sobre la evaluación del servicio:')
print('(restaurantes, usuarios) =', (UtMx_service.shape))
UtMx_service.head()

Dimensión de la matriz de Utilidad sobre la evaluación del servicio:
(restaurantes, usuarios) = (129, 138)


userID,U1001,U1002,U1003,U1004,U1005,U1006,U1007,U1008,U1009,U1010,...,U1129,U1130,U1131,U1132,U1133,U1134,U1135,U1136,U1137,U1138
name,,,,,,,,,,,,,,,,,,,,,
Abondance Restaurante Bar,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Arrachela Grill,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cabana Huasteca,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
Cafe Chaires,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cafeteria cenidet,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
# b. Incluye el código necesario para obtener y desplegar los 10 restaurantes con
#    mayor similaridad de Pearson al "Restaurante Pueblo Bonito", y usando solo
#    el 90% de las componentes que explican la varianza de la matriz UtMx_service.
#    Desplegarlos en orden descendente, sin incliur el restaurante de referencia.

# ************* Inlcuye aquí tu código:*****************************
 # Incluye todas las líneas de código o celdas que requieras.

#SVD con el máximo de componentes para calcular la varianza explicada
nc_service = min(UtMx_service.shape)
SVD_service = TruncatedSVD(n_components=nc_service, random_state=1)
SVD_service.fit_transform(UtMx_service)

#Número de componentes que explican el 90% de la variabilidad
for j in range(nc_service):
  if SVD_service.explained_variance_ratio_[0:j].sum() > 0.90:
    break
N_service = j-1
print('Valor de truncamiento al 90% de dicha variabilidad:', N_service)

#SVD truncada con N_service componentes
SVD_service = TruncatedSVD(n_components=N_service, random_state=1)
resultant_matrix_service = SVD_service.fit_transform(UtMx_service)

#Matriz de correlación de Pearson
corr_mat_service = np.corrcoef(resultant_matrix_service)

#Vector de correlación del restaurante de referencia
restaurante_de_referencia = "Restaurante Pueblo Bonito"
nombres_rest = UtMx_service.index
idx_rest = list(nombres_rest).index(restaurante_de_referencia)
corr_rest = corr_mat_service[idx_rest]

mejores_sim_service = [(corr_rest[i], nombres_rest[i])
                       for i in range(len(nombres_rest))
                       if corr_rest[i] > 0 and nombres_rest[i] != restaurante_de_referencia]

print('Total de similaridades positivas encontradas:', len(mejores_sim_service))

#Orden descendente y top 10
mejores_sim_service_ordenadas = sorted(mejores_sim_service, key=lambda simil: simil[0], reverse=True)

print('\nLos 10 restaurantes más similares a', restaurante_de_referencia,
      'con base en la evaluación del servicio:')
for k, (corr, nombre) in enumerate(mejores_sim_service_ordenadas[:10], start=1):
  print(f'{k:2d}. {nombre:45s} correlación = {corr:.4f}')

# *********** Aquí termina la sección de agregar código *************

Valor de truncamiento al 90% de dicha variabilidad: 51
Total de similaridades positivas encontradas: 54

Los 10 restaurantes más similares a Restaurante Pueblo Bonito con base en la evaluación del servicio:
 1. pizza clasica                                 correlación = 0.6620
 2. El Club                                       correlación = 0.5839
 3. Restaurante la Estrella de Dima               correlación = 0.4798
 4. Hamburguesas saul                             correlación = 0.4761
 5. Restaurante Guerra                            correlación = 0.4274
 6. Restaurant Orizatlan                          correlación = 0.4137
 7. la Cochinita Pibil Restaurante Yucateco       correlación = 0.3620
 8. Restaurante El Cielo Potosino                 correlación = 0.3498
 9. La Fontana Pizza Restaurante and Cafe         correlación = 0.3117
10. puesto de tacos                               correlación = 0.2994


## **Ejercicio - 7**

### **Incluye tus comentarios y conclusiones de la actividad. En particular indica cuáles restaurantes encontraste con correlaciones altas tanto para la calificación de la comida, como del servicio.**

++++++++ Inicia la sección de agregar texto: +++++++++++


###**Conclusión**

- Se construyeron dos matrices de utilidad (restaurantes × usuarios), una con la calificación de la comida y otra con la del servicio. A cada una se le aplicó una factorización SVD truncada y se conservaron solo las componentes que explican el 90% de la variabilidad (51 de 129 en el caso del servicio). Con las variables latentes resultantes se calculó la correlación de Pearson entre restaurantes para encontrar los más parecidos al Restaurante Pueblo Bonito.
- De los 10 más similares en cada criterio, 8 coinciden en ambas listas. Esto indica que los usuarios que califican de forma parecida a Pueblo Bonito tienden a hacerlo tanto en la comida como en el servicio.
- Los restaurantes con mayor similitud combinada son **pizza clasica, Restaurante la Estrella de Dima, El Club y Restaurante Guerra**, todos con promedio mayor a 0.5. Serían las mejores recomendaciones para alguien a quien le gustó Pueblo Bonito.
- Algunos solo aparecen en una lista. Pizzeria Julios y Restaurant Oriental Express son similares solo por la comida, y Restaurant Orizatlan y puesto de tacos solo por el servicio. Por eso conviene revisar más de un criterio antes de recomendar.
- La Estrella de Dima es la más parecida en comida (0.79), pero baja al tercer lugar en servicio (0.48). Las dos evaluaciones capturan aspectos distintos de la experiencia.
- Las correlaciones en general son moderadas (casi todas menores a 0.8), porque la matriz de utilidad es muy dispersa: cada usuario calificó pocos restaurantes. Aun así, la SVD permite encontrar patrones de similitud sin necesidad de etiquetas, lo que muestra la utilidad de este tipo de modelo no supervisado en sistemas de recomendación.
###**Restaurantes identificadas con Correlación Alta con Restaurante Pueblo Bonito**

| # | Nombre | Correlación Comida | Correlación Servicio | Promedio |
|---|--------|:------------------:|:--------------------:|:--------:|
| 1 | pizza clasica | 0.7036 | 0.6620 | 0.6828 |
| 2 | Restaurante la Estrella de Dima | 0.7942 | 0.4798 | 0.6370 |
| 3 | El Club | 0.5563 | 0.5839 | 0.5701 |
| 4 | Restaurante Guerra | 0.6543 | 0.4274 | 0.5409 |
| 5 | Hamburguesas saul | 0.3114 | 0.4761 | 0.3937 |
| 6 | la Cochinita Pibil Restaurante Yucateco | 0.2927 | 0.3620 | 0.3274 |
| 7 | Restaurante El Cielo Potosino | 0.2933 | 0.3498 | 0.3215 |
| 8 | La Fontana Pizza Restaurante and Cafe | 0.2930 | 0.3117 | 0.3023 |





++++++++ Termina la sección de agregar texto. +++++++++++

>> **Fin de la Actividad de Sistemas de Recomendación**